In [1]:
# 데이터 분석에 필요힌 라이브리리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#머신러닝 모델 사져오기
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

In [2]:
import cv2
import numpy as np

In [3]:
# pip install ultralytics #사용 하실때  압에 주석을 풀어서 사용해주요요

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
# pip install --upgrade typing_extensions

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

  Using cached typing_extensions-4.14.0-py3-none-any.whl.metadata (3.0 kB)
Using cached typing_extensions-4.14.0-py3-none-any.whl (43 kB)
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0


In [5]:
# pip install typing_extensions==4.11.0

Defaulting to user installation because normal site-packages is not writeable
  Using cached typing_extensions-4.11.0-py3-none-any.whl.metadata (3.0 kB)
Using cached typing_extensions-4.11.0-py3-none-any.whl (34 kB)
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.14.0
    Uninstalling typing_extensions-4.14.0:
      Successfully uninstalled typing_extensions-4.14.0
Note: you may need to restart the kernel to use updated packages.


In [6]:
# !pip install pytesseract #여가 까지 주것 해제 부분

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import os
# 여기는 스스
print(os.listdir(r"C:\Users\smhrd1\Desktop\ww\image"))

In [12]:
import pytesseract
# 여기는 스스
# Tesseract 실행 파일 경로 지정
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [13]:
import pytesseract
import os
#학인 부분
print("설정된 Tesseract 경로:", pytesseract.pytesseract.tesseract_cmd)
print("존재 여부:", os.path.isfile(pytesseract.pytesseract.tesseract_cmd))

설정된 Tesseract 경로: C:\Program Files\Tesseract-OCR\tesseract.exe
존재 여부: False


In [33]:
import platform
print(platform.architecture())
#학인 부분

('64bit', 'WindowsPE')


In [1]:
from ultralytics import YOLO
import cv2
import os
from pathlib import Path
import json
from datetime import datetime

# 기존 학습된 모델 사용
print("🚗 번호판 감지 시스템 시작!")
print("📚 학습된 모델 로드 중...")

# 학습이 완료된 모델 로드 (가장 최신 학습 결과 사용)
model = YOLO("yolo11n.pt")  # 또는 학습 후 생성된 best.pt 사용

# 결과 저장 경로 설정
output_base = r"D:\번호판"
os.makedirs(output_base, exist_ok=True)

# 하위 폴더들 생성
folders = {
    'detected_images': os.path.join(output_base, "감지결과_이미지"),
    'cropped_plates': os.path.join(output_base, "번호판_크롭"),
    'results_json': os.path.join(output_base, "결과_데이터"),
    'batch_results': os.path.join(output_base, "배치처리결과")
}

for folder in folders.values():
    os.makedirs(folder, exist_ok=True)

print(f"📁 결과 저장 경로: {output_base}")

# 1. 🔍 단일 이미지 번호판 감지 및 저장
def detect_license_plate_single(image_path, save_results=True):
    """단일 이미지에서 번호판 감지하고 결과 저장"""
    print(f"\n🔍 번호판 감지 중: {os.path.basename(image_path)}")
    
    # 이미지 감지
    results = model(image_path)
    
    # 원본 이미지 읽기
    original_img = cv2.imread(image_path)
    img_name = Path(image_path).stem
    
    detections = []
    plate_count = 0
    
    # 감지된 객체들 처리
    for i, box in enumerate(results[0].boxes):
        # 신뢰도 체크 (0.5 이상만)
        confidence = float(box.conf[0])
        if confidence < 0.5:
            continue
            
        # 바운딩 박스 좌표
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        class_id = int(box.cls[0])
        class_name = results[0].names[class_id] if results[0].names else f"class_{class_id}"
        
        # 번호판으로 추정되는 객체 또는 모든 감지된 객체
        detection_info = {
            "번호": i + 1,
            "클래스": class_name,
            "신뢰도": f"{confidence:.3f}",
            "좌표": {"x1": x1, "y1": y1, "x2": x2, "y2": y2}
        }
        detections.append(detection_info)
        
        # 번호판 영역 크롭하여 저장
        if save_results:
            cropped = original_img[y1:y2, x1:x2]
            if cropped.size > 0:  # 유효한 크기인지 확인
                crop_filename = f"{img_name}_번호판_{i+1}_{confidence:.3f}.jpg"
                crop_path = os.path.join(folders['cropped_plates'], crop_filename)
                cv2.imwrite(crop_path, cropped)
                print(f"  💾 번호판 크롭 저장: {crop_filename}")
                plate_count += 1
    
    # 감지 결과가 표시된 이미지 저장
    if save_results and len(detections) > 0:
        result_filename = f"{img_name}_감지결과.jpg"
        result_path = os.path.join(folders['detected_images'], result_filename)
        results[0].save(result_path)
        print(f"  🖼️  감지 결과 이미지 저장: {result_filename}")
    
    # JSON 결과 저장
    if save_results:
        json_data = {
            "파일명": os.path.basename(image_path),
            "처리시간": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "총_감지수": len(detections),
            "번호판_크롭수": plate_count,
            "감지결과": detections
        }
        
        json_filename = f"{img_name}_결과.json"
        json_path = os.path.join(folders['results_json'], json_filename)
        
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, indent=2, ensure_ascii=False)
        print(f"  📄 JSON 결과 저장: {json_filename}")
    
    print(f"✅ 감지 완료: {len(detections)}개 객체 발견, {plate_count}개 번호판 크롭")
    return detections

# 2. 📁 폴더 내 모든 이미지 배치 처리
def batch_detect_license_plates(input_folder):
    """폴더 내 모든 이미지에서 번호판 감지"""
    print(f"\n📁 배치 처리 시작: {input_folder}")
    
    # 지원하는 이미지 형식
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp']
    
    # 이미지 파일 찾기
    image_files = []
    for ext in image_extensions:
        image_files.extend(Path(input_folder).glob(f"*{ext}"))
        image_files.extend(Path(input_folder).glob(f"*{ext.upper()}"))
    
    if not image_files:
        print("❌ 이미지 파일을 찾을 수 없습니다!")
        return
    
    print(f"📊 총 {len(image_files)}개 이미지 발견")
    
    # 배치 처리 결과 저장용
    batch_summary = {
        "처리시간": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "총_이미지수": len(image_files),
        "처리결과": []
    }
    
    total_detections = 0
    total_plates = 0
    
    # 각 이미지 처리
    for i, img_path in enumerate(image_files, 1):
        print(f"\n📷 [{i}/{len(image_files)}] 처리중: {img_path.name}")
        
        try:
            detections = detect_license_plate_single(str(img_path), save_results=True)
            plates_found = len([d for d in detections if float(d['신뢰도']) > 0.5])
            
            total_detections += len(detections)
            total_plates += plates_found
            
            # 배치 요약에 추가
            batch_summary["처리결과"].append({
                "파일명": img_path.name,
                "감지수": len(detections),
                "번호판수": plates_found,
                "상태": "성공"
            })
            
        except Exception as e:
            print(f"  ❌ 처리 실패: {str(e)}")
            batch_summary["처리결과"].append({
                "파일명": img_path.name,
                "감지수": 0,
                "번호판수": 0,
                "상태": f"실패: {str(e)}"
            })
    
    # 배치 처리 요약 저장
    batch_summary.update({
        "총_감지수": total_detections,
        "총_번호판수": total_plates,
        "성공률": f"{len([r for r in batch_summary['처리결과'] if r['상태'] == '성공']) / len(image_files) * 100:.1f}%"
    })
    
    summary_path = os.path.join(folders['batch_results'], f"배치처리요약_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(batch_summary, f, indent=2, ensure_ascii=False)
    
    print(f"\n🎉 배치 처리 완료!")
    print(f"📊 처리 결과:")
    print(f"  - 총 이미지: {len(image_files)}개")
    print(f"  - 총 감지: {total_detections}개")
    print(f"  - 번호판 크롭: {total_plates}개")
    print(f"  - 요약 파일: {summary_path}")

# 3. 🎥 비디오에서 번호판 감지
def detect_license_plate_video(video_path, output_video_path=None):
    """비디오에서 번호판 감지 (실시간 처리)"""
    if output_video_path is None:
        video_name = Path(video_path).stem
        output_video_path = os.path.join(folders['batch_results'], f"{video_name}_번호판감지.mp4")
    
    print(f"\n🎥 비디오 번호판 감지: {os.path.basename(video_path)}")
    
    cap = cv2.VideoCapture(video_path)
    
    # 비디오 정보
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # 비디오 저장 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    frame_count = 0
    detection_log = []
    
    print(f"📹 비디오 정보: {width}x{height}, {fps}fps, {total_frames}프레임")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        # YOLO로 감지 (매 5프레임마다 처리로 속도 최적화)
        if frame_count % 5 == 0:
            results = model(frame)
            
            # 감지된 번호판 수 기록
            detections_count = len([box for box in results[0].boxes if float(box.conf[0]) > 0.5])
            if detections_count > 0:
                detection_log.append({
                    "프레임": frame_count,
                    "시간": f"{frame_count/fps:.2f}초",
                    "감지수": detections_count
                })
            
            # 결과를 프레임에 그리기
            annotated_frame = results[0].plot()
        else:
            annotated_frame = frame
        
        # 프레임 저장
        out.write(annotated_frame)
        
        # 진행상황 출력
        if frame_count % (fps * 5) == 0:  # 5초마다
            progress = (frame_count / total_frames) * 100
            print(f"  진행률: {progress:.1f}% ({frame_count}/{total_frames})")
    
    cap.release()
    out.release()
    
    # 감지 로그 저장
    video_name = Path(video_path).stem
    log_path = os.path.join(folders['results_json'], f"{video_name}_비디오감지로그.json")
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump({
            "비디오파일": os.path.basename(video_path),
            "총프레임": total_frames,
            "감지기록": detection_log,
            "총감지횟수": len(detection_log)
        }, f, indent=2, ensure_ascii=False)
    
    print(f"✅ 비디오 처리 완료!")
    print(f"  - 출력 비디오: {output_video_path}")
    print(f"  - 감지 로그: {log_path}")
    print(f"  - 번호판 감지 횟수: {len(detection_log)}회")

# 4. 📊 결과 리포트 생성
def generate_summary_report():
    """전체 처리 결과 리포트 생성"""
    print(f"\n📊 결과 리포트 생성 중...")
    
    # 각 폴더의 파일 수 계산
    report = {
        "생성시간": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "저장경로": output_base,
        "처리결과": {
            "감지결과_이미지": len(list(Path(folders['detected_images']).glob("*.jpg"))),
            "번호판_크롭": len(list(Path(folders['cropped_plates']).glob("*.jpg"))),
            "JSON_결과파일": len(list(Path(folders['results_json']).glob("*.json"))),
            "배치처리_결과": len(list(Path(folders['batch_results']).glob("*")))
        }
    }
    
    report_path = os.path.join(output_base, f"전체리포트_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    
    print(f"📄 리포트 저장: {report_path}")
    
    # 콘솔에 요약 출력
    print(f"\n📈 처리 결과 요약:")
    for key, value in report["처리결과"].items():
        print(f"  - {key}: {value}개")

# 5. 🚀 메인 실행 함수
def main():
    """메인 실행 함수"""
    print("="*60)
    print("🚗 YOLO 번호판 감지 시스템")
    print("="*60)
    
    # 기본 테스트 이미지로 시작
    test_image = r"C:\Users\smhrd1\Desktop\ww\image"
    
    if os.path.exists(test_image):
        if os.path.isfile(test_image):
            # 단일 파일인 경우
            print("📷 단일 이미지 테스트")
            detect_license_plate_single(test_image)
        else:
            # 폴더인 경우
            print("📁 폴더 배치 처리")
            batch_detect_license_plates(test_image)
    else:
        print(f"❌ 경로를 찾을 수 없습니다: {test_image}")
        print("💡 다른 이미지 경로를 지정해주세요")
    
    # 결과 리포트 생성
    generate_summary_report()
    
    print(f"\n🎉 모든 작업 완료!")
    print(f"📁 결과 확인: {output_base}")
    print("\n📂 생성된 폴더들:")
    for name, path in folders.items():
        print(f"  - {name}: {path}")

# 실행
if __name__ == "__main__":
    main()

# 💡 추가 사용법 예제:
"""
# 특정 이미지 처리
detect_license_plate_single(r"C:\path\to\image.jpg")

# 특정 폴더 배치 처리  
batch_detect_license_plates(r"C:\path\to\images")

# 비디오 처리
detect_license_plate_video(r"C:\path\to\video.mp4")
"""

🚗 번호판 감지 시스템 시작!
📚 학습된 모델 로드 중...
📁 결과 저장 경로: D:\번호판
🚗 YOLO 번호판 감지 시스템
📁 폴더 배치 처리

📁 배치 처리 시작: C:\Users\smhrd1\Desktop\ww\image
📊 총 12개 이미지 발견

📷 [1/12] 처리중: care.jpg

🔍 번호판 감지 중: care.jpg

image 1/1 C:\Users\smhrd1\Desktop\ww\image\care.jpg: 640x384 2 cars, 211.8ms
Speed: 4.1ms preprocess, 211.8ms inference, 3.1ms postprocess per image at shape (1, 3, 640, 384)
  💾 번호판 크롭 저장: care_번호판_1_0.922.jpg
  🖼️  감지 결과 이미지 저장: care_감지결과.jpg
  📄 JSON 결과 저장: care_결과.json
✅ 감지 완료: 1개 객체 발견, 1개 번호판 크롭

📷 [2/12] 처리중: carq.jpg

🔍 번호판 감지 중: carq.jpg

image 1/1 C:\Users\smhrd1\Desktop\ww\image\carq.jpg: 640x384 2 cars, 165.9ms
Speed: 2.4ms preprocess, 165.9ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 384)
  💾 번호판 크롭 저장: carq_번호판_1_0.796.jpg
  💾 번호판 크롭 저장: carq_번호판_2_0.741.jpg
  🖼️  감지 결과 이미지 저장: carq_감지결과.jpg
  📄 JSON 결과 저장: carq_결과.json
✅ 감지 완료: 2개 객체 발견, 2개 번호판 크롭

📷 [3/12] 처리중: cars.jpg

🔍 번호판 감지 중: cars.jpg

image 1/1 C:\Users\smhrd1\Desktop\ww\image\cars.jpg: 640x384 2 cars, 168.1

'\n# 특정 이미지 처리\ndetect_license_plate_single(r"C:\\path\to\\image.jpg")\n\n# 특정 폴더 배치 처리  \nbatch_detect_license_plates(r"C:\\path\to\\images")\n\n# 비디오 처리\ndetect_license_plate_video(r"C:\\path\to\x0bideo.mp4")\n'

In [3]:
import cv2
import os
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image

def show_image(image, title="Image"):
    """
    이미지를 화면에 표시하는 함수
    """
    # OpenCV 이미지를 RGB로 변환 (matplotlib용)
    if len(image.shape) == 3:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    else:
        image_rgb = image
    
    plt.figure(figsize=(10, 6))
    plt.imshow(image_rgb)
    plt.title(title)
    plt.axis('off')
    plt.show()

def show_extracted_plates(plates_folder):
    """
    추출된 번호판들을 한번에 보여주는 함수
    """
    plate_files = [f for f in os.listdir(plates_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    if not plate_files:
        print("추출된 번호판이 없습니다.")
        return
    
    # 그리드로 표시
    cols = 3
    rows = (len(plate_files) + cols - 1) // cols
    
    plt.figure(figsize=(15, 5 * rows))
    
    for i, filename in enumerate(plate_files):
        img_path = os.path.join(plates_folder, filename)
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img_rgb)
        plt.title(filename, fontsize=8)
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

def extract_license_plates(input_folder, output_folder, model_path=None, show_results=True):
    """
    욜로 모델을 사용해 이미지에서 번호판을 감지하고 추출하는 함수
    
    Args:
        input_folder: 입력 이미지 폴더 경로
        output_folder: 추출된 번호판 저장 폴더 경로
        model_path: 욜로 모델 경로 (None이면 기본 모델 사용)
    """
    
    # 출력 폴더 생성
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # 욜로 모델 로드 (번호판 감지용 커스텀 모델이 있다면 경로 지정)
    if model_path:
        model = YOLO(model_path)
    else:
        # 기본 욜로 모델 사용 (번호판 전용 모델이 없는 경우)
        model = YOLO('yolov8n.pt')
    
    # 지원하는 이미지 확장자
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
    
    # 폴더 내 모든 이미지 처리
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(valid_extensions):
            image_path = os.path.join(input_folder, filename)
            
            # 이미지 로드
            image = cv2.imread(image_path)
            if image is None:
                print(f"이미지를 읽을 수 없습니다: {filename}")
                continue
            
            # 욜로 모델로 객체 감지
            results = model(image)
            
            plate_count = 0
            
            # 감지된 객체들 처리
            for result in results:
                boxes = result.boxes
                if boxes is not None:
                    for i, box in enumerate(boxes):
                        # 신뢰도 확인 (0.5 이상인 경우만)
                        confidence = box.conf[0]
                        if confidence < 0.5:
                            continue
                        
                        # 바운딩 박스 좌표 추출
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                        
                        # 번호판 영역 추출
                        plate_img = image[y1:y2, x1:x2]
                        
                        if plate_img.size > 0:
                            plate_count += 1
                            
                            # 파일명 생성 (원본파일명_plate_번호.jpg)
                            base_name = os.path.splitext(filename)[0]
                            plate_filename = f"{base_name}_plate_{plate_count}.jpg"
                            plate_path = os.path.join(output_folder, plate_filename)
                            
                            # 번호판 이미지 저장
                            cv2.imwrite(plate_path, plate_img)
                            print(f"번호판 추출 완료: {plate_filename}")
                            
                            # 결과 이미지 표시
                            if show_results:
                                show_image(plate_img, f"추출된 번호판: {plate_filename}")
            
            if plate_count == 0:
                print(f"번호판을 찾을 수 없습니다: {filename}")

def extract_plates_with_coordinates(input_folder, output_folder, coordinates_file=None, show_results=True):
    """
    이미 감지된 좌표 정보가 있는 경우 사용하는 함수
    
    Args:
        input_folder: 입력 이미지 폴더 경로
        output_folder: 추출된 번호판 저장 폴더 경로
        coordinates_file: 좌표 정보가 담긴 텍스트 파일 (선택사항)
        show_results: 결과 이미지 표시 여부
    """
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
    
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(valid_extensions):
            image_path = os.path.join(input_folder, filename)
            image = cv2.imread(image_path)
            
            if image is None:
                continue
            
            # 좌표 파일이 있는 경우 해당 파일에서 좌표 읽기
            coord_file = os.path.join(input_folder, os.path.splitext(filename)[0] + '.txt')
            
            if os.path.exists(coord_file):
                with open(coord_file, 'r') as f:
                    lines = f.readlines()
                
                plate_count = 0
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:  # class_id, x_center, y_center, width, height
                        # 욜로 형식에서 실제 좌표로 변환
                        h, w = image.shape[:2]
                        x_center, y_center, width, height = map(float, parts[1:5])
                        
                        x1 = int((x_center - width/2) * w)
                        y1 = int((y_center - height/2) * h)
                        x2 = int((x_center + width/2) * w)
                        y2 = int((y_center + height/2) * h)
                        
                        # 좌표 범위 확인
                        x1 = max(0, x1)
                        y1 = max(0, y1)
                        x2 = min(w, x2)
                        y2 = min(h, y2)
                        
                        # 번호판 영역 추출
                        plate_img = image[y1:y2, x1:x2]
                        
                        if plate_img.size > 0:
                            plate_count += 1
                            base_name = os.path.splitext(filename)[0]
                            plate_filename = f"{base_name}_plate_{plate_count}.jpg"
                            plate_path = os.path.join(output_folder, plate_filename)
                            cv2.imwrite(plate_path, plate_img)
                            print(f"번호판 추출 완료: {plate_filename}")
                            
                            # 결과 이미지 표시
                            if show_results:
                                show_image(plate_img, f"추출된 번호판: {plate_filename}")

# 메인 실행 코드
if __name__ == "__main__":
    # 폴더 경로 설정
    input_folder = r"D:\번호판\감지결과_이미지"
    output_folder = r"D:\번호판\추출된_번호판"
    
    print("번호판 추출을 시작합니다...")
    
    # 방법 1: 욜로 모델로 직접 감지하여 추출 (이미지 표시 포함)
    # extract_license_plates(input_folder, output_folder, show_results=True)
    
    # 방법 2: 이미 감지된 좌표 정보를 사용하여 추출 (이미지 표시 포함)
    extract_plates_with_coordinates(input_folder, output_folder, show_results=True)
    
    print("번호판 추출이 완료되었습니다!")
    
    # 추출된 모든 번호판을 한번에 보기
    print("추출된 번호판들을 표시합니다...")
    show_extracted_plates(output_folder)

번호판 추출을 시작합니다...
번호판 추출이 완료되었습니다!
추출된 번호판들을 표시합니다...
추출된 번호판이 없습니다.


In [ ]:
# from ultralytics import YOLO

# model = YOLO("yolo11n.pt")

# train_results = model.train(
#     data="coco8.yaml",  
#     epochs=30,  
#     imgsz=640,  
#     device="cpu",  
# )

# metrics = model.val()

# results = model(r"C:\Users\smhrd1\Desktop\ww\image")  

# results[0].show() 

# path = model.export(format="onnx")

New https://pypi.org/project/ultralytics/8.3.152 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.146  Python-3.9.12 torch-2.7.0+cpu CPU (12th Gen Intel Core(TM) i7-12700)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train23, nbs=64, nms=False, opset=None, optimize=False, optimize

train: Scanning C:\Users\smhrd1\차량 번호판\datasets\coco8\labels\train.cache... 4 images, 0 backgrounds, 0 corrupt: 10

val: Fast image access  (ping: 0.30.1 ms, read: 81.734.7 MB/s, size: 54.0 KB)



C:\Users\smhrd1\AppData\Roaming\Python\Python39\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning C:\Users\smhrd1\차량 번호판\datasets\coco8\labels\val.cache... 4 images, 0 backgrounds, 0 corrupt: 100%|█
C:\Users\smhrd1\AppData\Roaming\Python\Python39\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Plotting labels to runs\detect\train23\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000119, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\train23
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...


30 epochs completed in 0.037 hours.
Optimizer stripped from runs\detect\train23\weights\last.pt, 5.5MB
Optimizer stripped from runs\detect\train23\weights\best.pt, 5.5MB

Validating runs\detect\train23\weights\best.pt...
Ultralytics 8.3.146  Python-3.9.12 torch-2.7.0+cpu CPU (12th Gen Intel Core(TM) i7-12700)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<0


                   all          4         17      0.654      0.937      0.914      0.653
                person          3         10      0.623        0.7      0.673      0.333
                   dog          1          1      0.531          1      0.995      0.796
                 horse          1          2      0.575          1      0.995      0.676
              elephant          1          2       0.64      0.921      0.828      0.323
              umbrella          1          1      0.557          1      0.995      0.895
          potted plant          1          1      0.999          1      0.995      0.895
Speed: 1.7ms preprocess, 189.8ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to runs\detect\train23
Ultralytics 8.3.146  Python-3.9.12 torch-2.7.0+cpu CPU (12th Gen Intel Core(TM) i7-12700)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 312.9209.1 MB/s, size: 54.0 KB)


val: Scanning C:\Users\smhrd1\차량 번호판\datasets\coco8\labels\val.cache... 4 images, 0 backgrounds, 0 corrupt: 100%|█
C:\Users\smhrd1\AppData\Roaming\Python\Python39\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<0


                   all          4         17      0.654      0.937      0.914      0.653
                person          3         10      0.623        0.7      0.673      0.333
                   dog          1          1      0.531          1      0.995      0.796
                 horse          1          2      0.575          1      0.995      0.676
              elephant          1          2       0.64      0.921      0.828      0.323
              umbrella          1          1      0.557          1      0.995      0.895
          potted plant          1          1      0.999          1      0.995      0.895
Speed: 1.4ms preprocess, 171.6ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to runs\detect\train232

image 1/6 C:\Users\smhrd1\Desktop\ww\image\care.jpg: 640x384 1 car, 171.9ms
image 2/6 C:\Users\smhrd1\Desktop\ww\image\carf.png: 640x384 2 cars, 176.9ms
image 3/6 C:\Users\smhrd1\Desktop\ww\image\carq.jpg: 640x384 2 cars, 148.9ms
image 4/6 C:\Users\smhrd

In [ ]:
# import re

# d = "carq.jpg\\care.jpg\\carf.png\\carq.jpg\\cars.jpg\\carw.jpg"

# # 1. 파일명만 추출 (확장자 제외)
# filenames = re.findall(r'([^\\]+?)\.(jpg|png)', d)

# # 2. 중복 제거 + 확장자 제외
# unique_names = sorted(set([name for name, ext in filenames]))

# # 3. 하나의 단어로 연결
# single_word = ''.join(unique_names)

# print(single_word)